In [ ]:
import os
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c deepfake-detection-challengekaggle

In [ ]:
import zipfile
with zipfile.ZipFile("deepfake-detection-challenge.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/deepfake-detection-challenge")

In [ ]:
import os
import json
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm
from tensorflow.keras.optimizers import Adam

In [ ]:
data_path = "/content/deepfake_data"
train_video_path = os.path.join(data_path, "train_sample_videos")
test_video_path = os.path.join(data_path, "test_videos")
metadata_path = os.path.join(train_video_path, "metadata.json")

In [ ]:
output_path = "data"  # Output folder

real_dir = os.path.join(output_path, "real")
fake_dir = os.path.join(output_path, "fake")
os.makedirs(real_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)

In [ ]:
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

def extract_frame(video_path, save_path, frame_num=0):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(save_path, frame)
    cap.release()

# Process videos
for video_file, info in tqdm(metadata.items()):
    video_path = os.path.join(train_video_path, video_file)
    if not os.path.exists(video_path):
        print(f"Warning: {video_path} not found!")
        continue

    label = info["label"].lower()
    save_folder = real_dir if label == "real" else fake_dir
    save_path = os.path.join(save_folder, f"{video_file.split('.')[0]}_frame1.jpg")

    extract_frame(video_path, save_path)

print("Frame extraction complete!")

In [ ]:
# Define parameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
DATA_DIR = "data"  # Base directory containing 'real' and 'fake' subfolders

# Data Augmentation and Loading
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True  # Freeze base model for transfer learning

# Custom Head for Binary Classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
x = Dense(512, activation="relu")(x)
x = Dense(1, activation="sigmoid")(x)  # Binary classification (Real/Fake)

model = Model(inputs=base_model.input, outputs=x)

# Compile Model
model.compile(optimizer=Adam(learning_rate=0.0001), loss="binary_crossentropy", metrics=["accuracy"])

# Train Model
history = model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)

# Save Model
model.save("deepfake_xception.h5")

print("Training complete and model saved!")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training vs Validation Loss')

plt.show()

In [ ]:
model.summary()